In [ ]:
%pip install -q pandas numpy scipy scikit-learn matplotlib wordcloud spacy ftfy
!python -m spacy download en_core_web_sm


In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('dataminds.csv')
data.head()

In [ ]:
import html
import re
import unicodedata
from html.parser import HTMLParser

import pandas as pd
from ftfy import fix_text


_TAGS = set("""a abbr article aside b blockquote body br caption code dd del
div dl dt em figcaption figure footer h1 h2 h3 h4 h5 h6 head header hr html
i img li main nav ol p pre s script section small span strong style sub sup
table tbody td th thead title tr u ul""".split())
_BLOCKS = set("""article aside blockquote br dd div dl dt figure footer h1 h2
h3 h4 h5 h6 header hr li main nav ol p pre section table td th tr ul""".split())
_HTML_TAG = re.compile(
    r"</?(?:" + "|".join(sorted(_TAGS)) + r")\b[^>]*>", re.IGNORECASE
)
_URL = re.compile(r"\b(?:https?://|www\.)[^\s<>]+", re.IGNORECASE)
_EMAIL = re.compile(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b")
_HANDLE = re.compile(r"(?<!\w)(?:/?u/[A-Za-z0-9_-]+|@[A-Za-z0-9_]+)")
_MARKDOWN_LINK = re.compile(r"!?\[([^\]]*)\]\((?:https?://|www\.)[^\s)]*\)")
_RESIDUAL_ENCODING = re.compile(r"\ufffd|‚Ä|â€|ðŸ|Ã[\u0080-\u00bf]|Â[\u0080-\u00bf]")


class _HTMLText(HTMLParser):
    """Extract visible text, keeping unknown angle-bracketed expressions."""

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts = []
        self.hidden = 0

    def handle_starttag(self, tag, attrs):
        if tag in {"script", "style", "head"}:
            self.hidden += 1
        if self.hidden:
            return
        if tag in _BLOCKS:
            self.parts.append(" ")
        elif tag == "img":
            self.parts.append(" " + (dict(attrs).get("alt") or "") + " ")
        elif tag not in _TAGS:
            self.parts.append(self.get_starttag_text())

    def handle_endtag(self, tag):
        if tag in {"script", "style", "head"}:
            self.hidden = max(0, self.hidden - 1)
        elif not self.hidden:
            if tag in _BLOCKS:
                self.parts.append(" ")
            elif tag not in _TAGS:
                self.parts.append("</" + tag + ">")

    def handle_data(self, value):
        if not self.hidden:
            self.parts.append(value)


def clean_text(value):
    """Repair common encoding/HTML artifacts and invisible formatting.

    Missing or non-string entries become empty only in the new columns.
    ZWJ/ZWNJ and combining marks are retained for emoji and non-Latin scripts.
    Unknown words, acronyms, slang, and spelling remain unchanged.
    """
    if not isinstance(value, str):
        return ""
    if value.strip().lower() in {"[deleted]", "[removed]"}:
        return ""

    text = fix_text(value, unescape_html=False, remove_control_chars=False)
    # Decode HTML entities, including common double-encoded &amp;amp; forms.
    for _ in range(3):
        decoded = html.unescape(text)
        if decoded == text:
            break
        text = decoded
    text = fix_text(text, unescape_html=False, remove_control_chars=False)

    if _HTML_TAG.search(text):
        parser = _HTMLText()
        parser.feed(text)
        parser.close()
        text = "".join(parser.parts)

    chars = []
    for char in text:
        category = unicodedata.category(char)
        if char == "\u200b":  # zero-width space: keep a word boundary
            chars.append(" ")
        elif char.isspace() or category in {"Cc", "Cs"}:
            chars.append(" ")
        elif category == "Cf" and char not in {"\u200c", "\u200d"}:
            continue  # BOM, soft hyphen, bidi controls, word joiner, etc.
        else:
            chars.append(char)

    text = unicodedata.normalize("NFC", "".join(chars))
    return re.sub(r"\s+", " ", text).strip()


def analysis_text(cleaned, abbreviations=None):
    """Make a lowercase view; optional expansions must be user-confirmed.

    Punctuation, negation, numbers, and emoji remain. Do not generate n-grams
    across different rows. Use text_clean for reading and semantic analysis.
    """
    text = _MARKDOWN_LINK.sub(r"\1", cleaned)
    text = _EMAIL.sub(" ", text)
    text = _URL.sub(" ", text)
    text = _HANDLE.sub(" ", text)

    if abbreviations:
        mapping = {str(key).lower(): str(value) for key, value in abbreviations.items()}
        if any(not key for key in mapping):
            raise ValueError("Abbreviation keys must not be empty.")
        pattern = re.compile(
            r"(?<!\w)(?:" + "|".join(
                re.escape(key) for key in sorted(mapping, key=len, reverse=True)
            ) + r")(?!\w)",
            re.IGNORECASE,
        )
        text = pattern.sub(lambda match: mapping[match.group().lower()], text)

    return re.sub(r"\s+", " ", text).strip().lower()


def preprocess_dataframe(data, source="text_original", abbreviations=None):
    """Return a copy with two text views and explicit review reasons.

    No rows are dropped, reindexed, or reordered; the source stays unchanged.
    needs_review flags invalid/empty text and possible unrepaired encoding,
    not whether the meaning or a clinical risk level is clear or unclear.
    Duplicate flags mark all nonempty copies; no deduplication is automatic.
    """
    if source not in data.columns:
        raise KeyError(f"Column {source!r} was not found.")
    outputs = {
        "text_clean", "text_analysis", "clean_is_empty", "is_duplicate_clean",
        "needs_review", "review_reason", "encoding_artifacts", "analysis_is_empty",
    }
    if source in outputs:
        raise ValueError("Use the original source column, not an output column.")

    result = data.copy()
    result["text_clean"] = result[source].map(clean_text)
    result["text_analysis"] = result["text_clean"].map(
        lambda value: analysis_text(value, abbreviations)
    )
    return refresh_review_flags(result)


def refresh_review_flags(data):
    """Recompute review flags from the current cleaned columns.

    Use this after editing cleaned text, or to inspect an older needs_review
    flag. The text columns and row order are preserved. An absent encoding
    match alone never implies that a post is empty.
    """
    for column in ("text_clean", "text_analysis"):
        if column not in data.columns:
            raise KeyError(f"Column {column!r} was not found. Run preprocessing first.")

    result = data.copy()
    cleaned = result["text_clean"].map(lambda value: value if isinstance(value, str) else "")
    analysis = result["text_analysis"].map(lambda value: value if isinstance(value, str) else "")
    artifacts = cleaned.map(lambda value: tuple(dict.fromkeys(_RESIDUAL_ENCODING.findall(value))))
    checks = {
        "empty cleaned text": cleaned.str.strip().eq(""),
        "empty analysis text": analysis.str.strip().eq(""),
        "possible encoding artifact": artifacts.map(bool),
    }
    reasons = pd.Series("", index=result.index, dtype=object)
    needs_review = pd.Series(False, index=result.index, dtype=bool)
    for reason, mask in checks.items():
        reasons = reasons + mask.map({True: reason + "; ", False: ""})
        needs_review = needs_review | mask

    result["clean_is_empty"] = checks["empty cleaned text"]
    result["analysis_is_empty"] = checks["empty analysis text"]
    result["is_duplicate_clean"] = (
        ~result["clean_is_empty"] & cleaned.duplicated(keep=False)
    )
    result["needs_review"] = needs_review
    result["review_reason"] = reasons.str.rstrip("; ").replace("", "no current issue")
    result["encoding_artifacts"] = artifacts
    return result


# Run from the original text; preserve IDs, labels, splits, and other CSV columns.
data = preprocess_dataframe(data, source="text_original")

# Keep this as a pandas Series, with the same row index as data.
text = data["text_analysis"]

display(data[["text_original", "text_clean", "text_analysis"]].head())

summary = pd.Series({
    "Total rows": len(data),
    "Empty cleaned text": int(data["clean_is_empty"].sum()),
    "Empty analysis text": int(data["analysis_is_empty"].sum()),
    "Possible remaining encoding artifacts": int(data["encoding_artifacts"].map(bool).sum()),
    "Duplicate cleaned text (all copies)": int(data["is_duplicate_clean"].sum()),
    "Rows requiring review": int(data["needs_review"].sum()),
}, name="Count")
display(summary.to_frame())


## Meaningful word and phrase visualizations

The plots below use `data` from the cleaning cell above. Your original text, labels, splits, and rows stay intact. Duplicate copies are removed only from the plotting view by default.

- **Content-word cloud and prevalence bars:** readable lemmas, with generic words such as *just*, *feel*, *like*, and *think* excluded. Size/count reflects how many posts mention a word, so one repetitive post cannot dominate.
- **Phrase cloud and prevalence bars:** original contiguous phrases of 2-5 spaCy tokens. Stopwords stay inside phrases, negation is preserved, and phrases cannot cross punctuation or post boundaries.
- **Existing group sizes and vocabulary heatmap:** compare percentages within the four existing TBe/PBu groups. Terms are selected by the difference between their group mention rate and their rate in all other labelled groups.
- **Post-length histogram:** counts words before filtering.

These are **lexical mentions, not codebook tag counts**. For example, a mention of *angry* can be negated or refer to another person. Stressor, relationship, emotion, and response tag charts should use validated annotations with evidence. This code does not generate new labels.

Exact duplicate texts with conflicting existing group labels are excluded from group comparisons and counted in the audit. Rare terms are filtered using `MIN_POSTS`; nested phrases with almost identical support are reduced in the plots, while all retained candidates remain in the tables.


In [ ]:
"""Readable word/phrase clouds and group comparisons for MINDMINER posts.

Install in a notebook:
    %pip install -q pandas numpy scipy scikit-learn matplotlib wordcloud spacy ftfy
    !python -m spacy download en_core_web_sm

Then:
    from meaningful_text_visualizations import run_visualizations
    results = run_visualizations(data)

Or run: python meaningful_text_visualizations.py dataminds.csv --no-show

Counts describe language mentions, not codebook annotations, diagnoses, or
suicide risk. Negated and third-person mentions still count as mentions.
No external API is called. Original text, labels, splits, and rows are preserved.
"""

from collections import Counter
from pathlib import Path
import html
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ftfy import fix_text
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from wordcloud import WordCloud


# These suppress generic standalone words, not the original text or phrases.
# Edit before inspecting a held-out test set if using these features for ML.
GENERIC_WORDS = set("""
just really actually basically literally quite maybe probably anyway also
still even ever always already often sometimes somehow thing stuff something
anything everything someone somebody anyone anybody everyone everybody
get go come make take give put know think feel want need say tell seem look try
like life time day year week month people much many lot bit way point
im ive ill id youre youve youll theyre thats theres dont cant wont
edit update tldr tl dr reddit post read thank thanks lol lmao omg tbh
""".split())

NEGATIONS = {"no", "not", "never", "neither", "nor", "without", "none", "n't"}
CLITICS = {"n't", "'s", "'m", "'re", "'ve", "'ll", "'d"}
CONTENT_POS = {"NOUN", "PROPN", "VERB", "ADJ", "ADV"}
LEXICAL_STOP = STOP_WORDS - {"alone", "empty"}
PHRASE_FILLERS = set("just really actually basically literally tbh lol lmao".split())
EXPERIENCE_VERBS = set("""
feel need want seek lose lack miss hate love worry struggle suffer ask talk
help end afford cope find stop
""".split())
GROUP_ORDER = ["Neither", "TBe only", "PBu only", "Both"]
STYLE = {
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.titlesize": 13, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
}


def _normalise(value):
    if not isinstance(value, str):
        return ""
    value = html.unescape(fix_text(value)).replace("’", "'")
    if value.strip().lower() in {"[removed]", "[deleted]"}:
        return ""
    # Keep link labels, but replace identifiers/URLs with a phrase boundary.
    value = re.sub(r"\[([^\]]*)\]\(https?://[^\s)]*\)", r"\1", value)
    value = re.sub(r"\b(?:https?://|www\.)\S+", " . ", value)
    value = re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", " . ", value)
    value = re.sub(r"(?<!\w)(?:/?u/[A-Za-z0-9_-]+|@[A-Za-z0-9_]+)", " . ", value)
    return re.sub(r"\s+", " ", value).strip()


def _existing_groups(data):
    # Reconstruct the four groups from existing labels; never infer from text.
    if {"belong", "burden"}.issubset(data.columns):
        belong = pd.to_numeric(data["belong"], errors="coerce")
        burden = pd.to_numeric(data["burden"], errors="coerce")
        mapping = {(0, 0): "Neither", (1, 0): "TBe only",
                   (0, 1): "PBu only", (1, 1): "Both"}
        return [mapping.get((b, p)) for b, p in zip(belong, burden)]
    if "group" in data:
        return [v if isinstance(v, str) and v in GROUP_ORDER else None
                for v in data["group"]]
    return [None] * len(data)


def _prepare(data, text_column, deduplicate):
    if not isinstance(data, pd.DataFrame):
        raise TypeError("Pass your DataFrame, e.g. run_visualizations(data).")
    if text_column is None:
        text_column = next((c for c in ("text_clean", "text_analysis", "text", "text_original")
                            if c in data), None)
    if text_column not in data:
        raise KeyError("Set text_column to the name of your text column.")
    work = pd.DataFrame({
        "source_row_index": list(data.index),
        "post_id": data["post_id"].tolist() if "post_id" in data else list(range(len(data))),
        "_text": data[text_column].map(_normalise).tolist(),
        "_group": _existing_groups(data),
    })
    # Use cleaned full text for exact-duplicate identity, not a filtered bag of words.
    key_source = data["text_clean"] if "text_clean" in data else data[text_column]
    work["_key"] = key_source.map(_normalise).tolist()
    empty_count = int(work["_text"].eq("").sum())
    work = work.loc[work["_text"].ne("")].copy()
    if work.empty:
        raise ValueError("No usable text was found.")
    labelled = work.loc[work["_group"].notna()]
    group_counts = labelled.groupby("_key")["_group"].nunique()
    conflicts = set(group_counts.index[group_counts.gt(1)])
    duplicate_count = int(work.duplicated("_key").sum())
    if deduplicate:
        # A labelled copy can supply a group even if the first copy is unlabelled.
        known_groups = labelled.drop_duplicates("_key").set_index("_key")["_group"]
        work = work.drop_duplicates("_key", keep="first").copy()
        work["_group"] = work["_key"].map(known_groups)
    conflict_mask = work["_key"].isin(conflicts)
    conflict_rows = int(conflict_mask.sum())
    work.loc[conflict_mask, "_group"] = None
    work = work.reset_index(drop=True)
    audit = {
        "input_rows": len(data), "text_column": text_column,
        "empty_rows_excluded": empty_count,
        "extra_duplicate_copies": duplicate_count,
        "duplicate_copies_removed_for_plots": duplicate_count if deduplicate else 0,
        "rows_used_for_plots": len(work),
        "duplicate_texts_with_conflicting_groups": len(conflicts),
        "retained_conflicting_rows_excluded_from_group_plots": conflict_rows,
        "rows_without_usable_group": int(work["_group"].isna().sum()),
    }
    return work, audit


def _is_anchor(token):
    return (token.is_alpha and token.pos_ in CONTENT_POS
            and token.lower_ not in LEXICAL_STOP
            and token.lower_ not in PHRASE_FILLERS)


def extract_features(doc, generic_words):
    """Unigrams use lemmas; phrases use contiguous ORIGINAL surface tokens.

    Stopwords are NOT removed before building n-grams. A phrase cannot cross
    a sentence, punctuation, URL boundary, or post. This avoids fake phrases
    such as 'not friend' from 'not talking to my friend'. Phrase rules select
    lexical candidates for exploration, not semantic tags.
    """
    words, phrases = [], []
    for token in doc:
        lemma = token.lemma_.lower()
        if (_is_anchor(token) and len(lemma) >= 3
                and lemma not in LEXICAL_STOP and lemma not in generic_words
                and token.lower_ not in generic_words and lemma not in NEGATIONS):
            words.append(lemma)
    for sentence in doc.sents:
        tokens = list(sentence)
        for start in range(len(tokens)):
            for size in range(2, 6):
                window = tokens[start:start + size]
                if len(window) < size:
                    break
                if any(not (t.is_alpha or t.lower_ in CLITICS) for t in window):
                    break  # Increasing the window cannot undo a punctuation boundary.
                first, last = window[0], window[-1]
                has_negation = any(t.lower_ in NEGATIONS for t in window)
                valid_start = (_is_anchor(first) or first.lower_ in NEGATIONS
                               or (first.pos_ == "AUX" and has_negation))
                if first.lower_ in CLITICS or not valid_start or not _is_anchor(last):
                    continue
                # Avoid fragments ending before their noun/object, e.g.
                # "need professional" from "need professional help".
                if last.dep_ in {"amod", "compound"} and last.head.i > last.i:
                    continue
                if last.pos_ in {"VERB", "AUX"} and any(
                    child.dep_ in {"dobj", "obj", "attr", "acomp", "xcomp", "ccomp"}
                    and child.i > last.i for child in last.children
                ):
                    continue
                # Do not bridge coordinated verb clauses into one phrase.
                if any(t.dep_ == "cc" and t.head.pos_ in {"VERB", "AUX"} for t in window):
                    continue
                # Do not also show a positive-looking subphrase such as
                # "feel connected" taken from "don't feel connected".
                # Retain the span only when directly attached negation is inside it.
                if any(child.dep_ == "neg" and not (first.i <= child.i <= last.i)
                       for token in window for child in token.children):
                    continue
                if any(t.lower_ in PHRASE_FILLERS for t in window):
                    continue
                informative = sum(_is_anchor(t) and t.lemma_.lower() not in generic_words
                                  and t.lower_ not in generic_words for t in window)
                if informative < 2 and not (
                    informative >= 1 and (has_negation or first.lemma_.lower() in EXPERIENCE_VERBS)
                ):
                    continue
                phrase = doc[first.i:last.i + 1].text.lower()
                phrase = re.sub(r"\s+", " ", phrase).strip()
                if len(phrase) <= 65:
                    phrases.append(phrase)
    return words, phrases, sum(t.is_alpha for t in doc)


def _identity(tokens):
    return tokens


def _matrix_and_table(documents, min_posts, max_fraction):
    n = len(documents)
    counts = Counter(term for terms in documents for term in set(terms))
    vocab = sorted(term for term, count in counts.items()
                   if min_posts <= count <= max_fraction * n)
    columns = ["term", "post_count", "post_percent", "occurrences", "mean_tfidf"]
    if not vocab:
        return sparse.csr_matrix((n, 0), dtype=np.int32), np.array([], dtype=str), pd.DataFrame(columns=columns)
    vectorizer = CountVectorizer(analyzer=_identity, lowercase=False, vocabulary=vocab)
    matrix = vectorizer.fit_transform(documents)
    names = vectorizer.get_feature_names_out()
    df = np.asarray((matrix > 0).sum(axis=0)).ravel()
    tfidf = TfidfTransformer(sublinear_tf=True).fit_transform(matrix)
    table = pd.DataFrame({
        "term": names, "post_count": df.astype(int), "post_percent": df / n * 100,
        "occurrences": np.asarray(matrix.sum(axis=0)).ravel().astype(int),
        "mean_tfidf": np.asarray(tfidf.mean(axis=0)).ravel(),
    }).sort_values(["post_count", "term"], ascending=[False, True], ignore_index=True)
    return matrix, names, table


def _phrase_display_table(table, limit, rank_by="post_count"):
    # Hide near-identical nested phrases in plots only; retain all in the CSV.
    selected = []
    ranked = table.assign(_length=table["term"].str.len()).sort_values(
        [rank_by, "_length", "term"], ascending=[False, False, True])
    for _, row in ranked.iterrows():
        redundant = any(
            (f" {row['term']} " in f" {old['term']} "
             or f" {old['term']} " in f" {row['term']} ")
            and min(row["post_count"], old["post_count"]) /
                max(row["post_count"], old["post_count"]) >= 0.85
            and bool(re.search(r"\b(?:no|not|never|without|none|neither|nor)\b|n't\b", row["term"])) ==
                bool(re.search(r"\b(?:no|not|never|without|none|neither|nor)\b|n't\b", old["term"]))
            for old in selected
        )
        if not redundant:
            selected.append(row)
        if len(selected) >= limit:
            break
    return pd.DataFrame(selected, columns=table.columns)


def _bars(table, title, color):
    fig, ax = plt.subplots(figsize=(10, max(4, 0.32 * len(table) + 1.6)))
    if table.empty:
        ax.text(.5, .5, "No terms meet min_posts; lower it for a small sample.",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
    else:
        values = table["post_percent"].to_numpy(float)
        positions = np.arange(len(table))
        ax.barh(positions, values, color=color)
        ax.set_yticks(positions, table["term"])
        ax.invert_yaxis()
        ax.set_xlim(0, max(1, values.max() * 1.32))
        for y, (_, row) in enumerate(table.iterrows()):
            ax.text(float(row["post_percent"]) + values.max() * .015, y,
                    f"{row['post_percent']:.1f}%  (n={int(row['post_count'])})", va="center", fontsize=9)
        ax.set_xlabel("Posts mentioning the term (%)")
        ax.grid(axis="x", alpha=.18)
        ax.set_axisbelow(True)
    ax.set_title(title, loc="left", pad=14)
    fig.tight_layout()
    return fig


def _group_tables(matrix, names, types, groups, min_group_posts, min_term_posts):
    rows = []
    present = groups.notna().to_numpy()
    binary = matrix > 0
    for group in GROUP_ORDER:
        inside = groups.eq(group).to_numpy()
        outside = present & ~inside
        ng, nr = int(inside.sum()), int(outside.sum())
        if ng < min_group_posts or nr < min_group_posts or matrix.shape[1] == 0:
            continue
        cg = np.asarray(binary[inside].sum(axis=0)).ravel()
        cr = np.asarray(binary[outside].sum(axis=0)).ravel()
        delta = 100 * (cg / ng - cr / nr)
        for j in np.flatnonzero((cg >= min_term_posts) & (delta > 0)):
            rows.append({
                "group": group, "term": names[j], "term_type": types[j],
                "group_post_count": int(cg[j]), "group_n": ng,
                "group_percent": 100 * cg[j] / ng,
                "other_post_count": int(cr[j]), "other_n": nr,
                "other_percent": 100 * cr[j] / nr,
                "difference_percentage_points": delta[j],
            })
    columns = ["group", "term", "term_type", "group_post_count", "group_n",
               "group_percent", "other_post_count", "other_n", "other_percent",
               "difference_percentage_points"]
    table = pd.DataFrame(rows, columns=columns)
    return table.sort_values(["group", "difference_percentage_points", "group_post_count", "term"],
                             ascending=[True, False, False, True], ignore_index=True)


def run_visualizations(data, text_column=None, output_dir="meaningful_text_visualizations",
                       min_posts=5, max_post_fraction=.80, top_n=20,
                       max_cloud_words=60, extra_generic_words=None,
                       deduplicate=True, min_group_posts=10, show=True,
                       nlp=None, font_path=None):
    """Return figures/tables and save PNGs + CSVs; data is never modified.

    min_posts is a minimum number of POSTS, not token occurrences.
    Generic-word removal and the document-frequency ceiling affect displays.
    Groups use existing belong/burden labels (or the exact group strings).
    Duplicate texts with conflicting groups are excluded from group plots.
    Group differences are descriptive; no significance or causal claim is made.
    nlp can be an already-loaded en_core_web_sm pipeline with its parser active.
    """
    if not isinstance(min_posts, int) or min_posts < 1:
        raise ValueError("min_posts must be a positive integer.")
    if not 0 < max_post_fraction <= 1:
        raise ValueError("max_post_fraction must be in (0, 1].")
    if top_n < 1 or max_cloud_words < 1 or min_group_posts < 1:
        raise ValueError("Plot limits and min_group_posts must be positive.")
    work, audit = _prepare(data, text_column, deduplicate)
    if nlp is None:
        try:
            nlp = spacy.load("en_core_web_sm", disable=["ner"])
        except OSError as exc:
            raise RuntimeError("Install the model first: python -m spacy download en_core_web_sm") from exc
    generic = GENERIC_WORDS | {str(w).lower() for w in (extra_generic_words or set())}
    word_docs, phrase_docs, lengths = [], [], []
    print(f"Processing {len(work):,} posts; original DataFrame is preserved.")
    for doc in nlp.pipe(work["_text"].tolist(), batch_size=64):
        if not doc.has_annotation("POS") or not doc.has_annotation("LEMMA") or not doc.has_annotation("SENT_START"):
            raise ValueError("Use en_core_web_sm with tagger, lemmatizer and parser active.")
        words, phrases, length = extract_features(doc, generic)
        word_docs.append(words)
        phrase_docs.append(phrases)
        lengths.append(length)
    word_x, word_names, word_table = _matrix_and_table(word_docs, min_posts, max_post_fraction)
    phrase_x, phrase_names, phrase_table = _matrix_and_table(phrase_docs, min_posts, max_post_fraction)
    all_x = sparse.hstack([word_x, phrase_x], format="csr")
    all_names = np.concatenate([word_names, phrase_names])
    all_types = np.array(["word"] * len(word_names) + ["phrase"] * len(phrase_names))
    group_terms = _group_tables(all_x, all_names, all_types, work["_group"],
                               min_group_posts, min_posts)
    groups = work["_group"].value_counts().reindex(GROUP_ORDER, fill_value=0)
    group_sizes = groups.rename_axis("group").rename("posts").reset_index()
    group_sizes["percent_of_labelled_posts"] = group_sizes["posts"] / max(1, groups.sum()) * 100
    length_table = work[["source_row_index", "post_id"]].copy()
    length_table["word_count"] = lengths
    figures = {}
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(17, 6))
        for ax, table, title, cmap, limit in [
            (axes[0], word_table, "Content words (lemmatized)", "viridis", max_cloud_words),
            (axes[1], _phrase_display_table(phrase_table, 35), "Original phrases (2-5 tokens)", "cividis", 35),
        ]:
            if table.empty:
                ax.text(.5, .5, "No terms meet the frequency filters.", ha="center", va="center")
            else:
                cloud = WordCloud(width=1400, height=900, background_color="white",
                                  max_words=limit, min_font_size=12, prefer_horizontal=1,
                                  collocations=False, random_state=42, colormap=cmap,
                                  font_path=font_path).generate_from_frequencies(
                                      dict(zip(table["term"], table["post_count"])))
                ax.imshow(cloud, interpolation="bilinear")
            ax.set_axis_off()
            ax.set_title(title, loc="left")
        fig.suptitle(f"Language across {len(work):,} posts | size = number of posts mentioning a term", y=1.01)
        fig.tight_layout()
        figures["01_word_and_phrase_clouds"] = fig
        figures["02_word_prevalence"] = _bars(word_table.head(top_n), "Most widespread content words", "#236B80")
        figures["03_phrase_prevalence"] = _bars(_phrase_display_table(phrase_table, top_n),
                                               "Repeated phrases with their wording preserved", "#54804A")
        fig, ax = plt.subplots(figsize=(9, 4.5))
        ax.hist(lengths, bins=min(40, max(1, int(np.sqrt(len(work))))), color="#236B80", alpha=.9)
        median = float(np.median(lengths))
        ax.axvline(median, color="#AD5133", ls="--", label=f"Median: {median:,.0f} words")
        ax.set(title="Post lengths before stopword removal", xlabel="spaCy alphabetic word tokens per post", ylabel="Posts")
        ax.legend(frameon=False)
        fig.tight_layout()
        figures["04_post_lengths"] = fig
        if groups.sum() > 0:
            fig, ax = plt.subplots(figsize=(8, 4.5))
            bars = ax.bar(GROUP_ORDER, groups, color=["#8194A1", "#236B80", "#B56B43", "#54804A"])
            ax.bar_label(bars, padding=4, fmt="%.0f")
            ax.set_ylim(0, max(1, groups.max() * 1.16))
            ax.set(title="Existing TBe/PBu group sizes", ylabel="Posts with usable existing group labels")
            fig.tight_layout()
            figures["05_existing_group_sizes"] = fig
        if not group_terms.empty:
            # Up to two words + two phrases per group; positive prevalence gaps.
            chosen = []
            for group in GROUP_ORDER:
                for kind in ("word", "phrase"):
                    part = group_terms.loc[group_terms["group"].eq(group) & group_terms["term_type"].eq(kind)]
                    if kind == "phrase":
                        # Reduce nested phrases within each group's display.
                        part = _phrase_display_table(
                            part.rename(columns={"group_post_count": "post_count"}), 2,
                            rank_by="difference_percentage_points")
                    chosen.extend(part.head(2)["term"].tolist())
            chosen = list(dict.fromkeys(chosen))
            name_to_column = {name: i for i, name in enumerate(all_names)}
            selected = all_x[:, [name_to_column[name] for name in chosen]] > 0
            active_groups = [g for g in GROUP_ORDER if groups[g] > 0]
            prevalence = np.column_stack([
                np.asarray(selected[work["_group"].eq(g).to_numpy()].mean(axis=0)).ravel() * 100
                for g in active_groups
            ])
            fig, ax = plt.subplots(figsize=(10, max(4.5, .36 * len(chosen) + 2)))
            im = ax.imshow(prevalence, cmap="Blues", vmin=0, vmax=max(1, prevalence.max()), aspect="auto")
            ax.set_yticks(range(len(chosen)), chosen)
            ax.set_xticks(range(len(active_groups)), [f"{g}\n(n={int(groups[g]):,})" for g in active_groups])
            threshold = prevalence.max() * .55
            for i in range(len(chosen)):
                for j in range(len(active_groups)):
                    ax.text(j, i, f"{prevalence[i,j]:.1f}%", ha="center", va="center", fontsize=9,
                            color="white" if prevalence[i,j] > threshold else "#172A38")
            ax.set_title("Terms more prevalent in at least one existing group\nCells show within-group mention rates; selection uses group minus other groups", loc="left", pad=15)
            fig.colorbar(im, ax=ax, label="Posts mentioning term within group (%)", shrink=.75)
            fig.tight_layout()
            figures["06_group_language_heatmap"] = fig
    audit.update({"min_posts": min_posts, "max_post_fraction": max_post_fraction,
                  "word_features": len(word_names), "phrase_features": len(phrase_names),
                  "min_group_posts_for_term_comparison": min_group_posts})
    tables = {"word_terms": word_table, "phrase_terms": phrase_table,
              "group_distinctive_terms": group_terms, "existing_group_sizes": group_sizes,
              "post_lengths": length_table}
    if output_dir is not None:
        folder = Path(output_dir)
        folder.mkdir(parents=True, exist_ok=True)
        for name, figure in figures.items():
            figure.savefig(folder / f"{name}.png", dpi=300, bbox_inches="tight")
        for name, table in tables.items():
            table.to_csv(folder / f"{name}.csv", index=False, encoding="utf-8-sig")
        settings = {**audit, "generic_words": sorted(generic), "deduplicate_for_plots": deduplicate,
                    "spacy_model": nlp.meta.get("name"), "spacy_model_version": nlp.meta.get("version")}
        (folder / "analysis_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
        print(f"Saved {len(figures)} PNG figures, {len(tables)} CSV tables and settings to {folder.resolve()}")
    print(pd.Series(audit).to_string())
    print("Mention rates are lexical summaries, not codebook tag prevalence.")
    if group_terms.empty:
        print("Group term comparison skipped: no supported terms/groups meet the thresholds.")
    if show:
        plt.show()
    else:
        for figure in figures.values():
            plt.close(figure)
    return {"figures": figures, **tables, "audit": audit}


In [ ]:
MIN_POSTS = 5                     # A term must appear in at least this many posts.
MAX_POST_FRACTION = 0.80           # Exclude terms appearing in >80% of posts.
TOP_N = 20
EXTRA_GENERIC_WORDS = set()        # Add corpus-specific filler words if needed.
OUTPUT_DIR = "meaningful_text_visualizations"
DEDUPLICATE_FOR_PLOTS = True

plot_results = run_visualizations(
    data,
    text_column="text_clean",    # Uses your existing cleaned text, with punctuation.
    output_dir=OUTPUT_DIR,
    min_posts=MIN_POSTS,
    max_post_fraction=MAX_POST_FRACTION,
    top_n=TOP_N,
    extra_generic_words=EXTRA_GENERIC_WORDS,
    deduplicate=DEDUPLICATE_FOR_PLOTS,
    min_group_posts=10,
)


In [ ]:
# @title
# Inspect the exact values behind the charts.
print("Content-word mention rates")
display(plot_results["word_terms"].head(TOP_N))

print("Original-phrase mention rates (all retained candidates)")
display(plot_results["phrase_terms"].head(TOP_N))

print("Existing group sizes")
display(plot_results["existing_group_sizes"])

print("Terms more prevalent in each existing group")
if plot_results["group_distinctive_terms"].empty:
    print("No group comparison met the support thresholds.")
else:
    display(plot_results["group_distinctive_terms"].groupby("group", sort=False).head(8))

print("Text and duplicate audit")
display(pd.Series(plot_results["audit"], name="Value").to_frame())


### Reading and adjusting the results

`post_percent` is the percentage of retained posts containing a term at least once. `occurrences` counts all repetitions. The tables also include mean TF-IDF for optional inspection; the main plots use post mention rates for a clear interpretation.

`difference_percentage_points` compares a term's mention rate within one existing group against all other labelled groups. It describes an association in this sample, not a causal effect or statistical significance. If the corpus was balanced by sampling, group sizes and tag frequencies describe that sample rather than population prevalence.

For a small sample, reduce `MIN_POSTS` to 2 and `min_group_posts` to 5. For a noisier full corpus, increase `MIN_POSTS` to 10. Do not adjust these settings using a held-out test set when evaluating a prediction model.

Each run saves PNG charts, CSV tables, and an `analysis_settings.json` file in `OUTPUT_DIR`. No post text is exported. Codebook tag prevalence requires a separate table of evidence-supported annotations; keyword matches are insufficient.


In [ ]:
# @title
# Optional: create a ZIP of the generated charts and tables.
# Uncomment to run. In Colab, the last two lines download it.
# import shutil
# zip_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
# from google.colab import files
# files.download(zip_path)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

# Counts read from your image: Train + Validation + Test
counts = np.array([
    [909 + 214 + 471, 1062 + 279 + 584],  # Belong: 0, 1
    [1335 + 307 + 731, 636 + 186 + 324],  # Burden: 0, 1
])

totals = counts.sum(axis=1)
x, width = np.arange(2), 0.30

fig, ax = plt.subplots(figsize=(6.6, 4.2), layout="constrained")

for label, color in enumerate(["red", "#007BB5"]):
    values = counts[:, label]
    bars = ax.bar(
        x + (label - 0.5) * width,
        values,
        width=width,
        color=color,
        label=str(label),
    )
    ax.bar_label(
        bars,
        labels=[
            f"{value:,}\n({value / total:.1%})"
            for value, total in zip(values, totals)
        ],
        padding=5,
        fontsize=10,
    )

ax.set_xticks(x, ["Belong", "Burden"])
ax.set_ylabel("Posts")
ax.set_title("Train, validation and test combined (N = 3,519)")
ax.set_ylim(0, counts.max() * 1.22)
ax.margins(x=0.18)
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Label", ncol=2, frameon=False, loc="upper left")

fig.savefig("combined_belong_burden.png", dpi=300, bbox_inches="tight")
plt.show()